In [1]:
# Cell 1: Import libraries & setup
import sys
print(f"Python path: {sys.executable}")

from google.cloud import bigquery
import pandas as pd
import numpy as np

PROJECT_ID = "telco-portfolio"
DATASET_ID = "telco_dataset"
SOURCE_TABLE = "raw_churn"
TARGET_TABLE = "features_network_kqi"

client = bigquery.Client(project=PROJECT_ID)

print(f"✅ Connected to project: {client.project}")
print(f"✅ Pandas version: {pd.__version__}")
print(f"✅ Target table: {TARGET_TABLE}")

Python path: /Users/macos/Study_burhanudin_2025/GCP/telco-ai-portfolio/venv_telco/bin/python
✅ Connected to project: telco-portfolio
✅ Pandas version: 2.3.3
✅ Target table: features_network_kqi


In [4]:
# Cek tipe data kolom Churn
query_check = f"""
SELECT 
  column_name,
  data_type
FROM `{PROJECT_ID}.{DATASET_ID}.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = '{SOURCE_TABLE}' AND column_name = 'Churn'
"""

check = client.query(query_check).to_dataframe()
print(check)

/Users/macos/Study_burhanudin_2025/GCP/telco-ai-portfolio/venv_telco/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


  column_name data_type
0       Churn      BOOL


In [5]:
sql_create_features = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.{TARGET_TABLE}` AS

WITH base AS (
  SELECT 
    customerID,
    gender,
    SeniorCitizen,
    Partner,
    Dependents,
    tenure,
    PhoneService,
    MultipleLines,
    InternetService,
    OnlineSecurity,
    OnlineBackup,
    DeviceProtection,
    TechSupport,
    StreamingTV,
    StreamingMovies,
    Contract,
    PaperlessBilling,
    PaymentMethod,
    MonthlyCharges,
    TotalCharges,
    Churn
  FROM `{PROJECT_ID}.{DATASET_ID}.{SOURCE_TABLE}`
)

SELECT 
  *,
  
  -- ============================================
  -- NETWORK KQI SIMULATION (TELCO-SPECIFIC)
  -- ============================================
  
  -- 1. avg_throughput_mbps (higher is better)
  CASE 
    WHEN InternetService = 'Fiber optic' THEN 50 + (tenure/100)*20 + RAND()*15
    WHEN InternetService = 'DSL' THEN 25 + (tenure/100)*10 + RAND()*10
    ELSE 5 + RAND()*5
  END AS avg_throughput_mbps,
  
  -- 2. latency_ms (lower is better)
  CASE 
    WHEN InternetService = 'Fiber optic' THEN 20 - (tenure/200)*10 + RAND()*10
    WHEN InternetService = 'DSL' THEN 45 - (tenure/100)*5 + RAND()*15
    ELSE 100 + RAND()*50
  END AS latency_ms,
  
  -- 3. drop_call_rate (0-1, lower is better)
  0.01 + (1 - LEAST(1, tenure/72)) * 0.1 + (SeniorCitizen * 0.05) + RAND()*0.05 AS drop_call_rate,
  
  -- 4. network_quality_score (0-100, higher is better)
  -- PERBAIKAN: Churn adalah BOOL (TRUE/FALSE, bukan 'Yes'/'No')
  LEAST(100, GREATEST(0,
    ((CASE 
      WHEN InternetService = 'Fiber optic' THEN 80
      WHEN InternetService = 'DSL' THEN 60
      ELSE 30
    END) * 0.4) +
    (100 - LEAST(100, (tenure/72)*100)) * 0.2 +
    ((1 - SeniorCitizen) * 20) * 0.2 +
    (CASE WHEN Churn = FALSE THEN 20 ELSE 0 END) * 0.2
  )) AS network_quality_score,
  
  -- 5. customer_segment (based on tenure & monthly charges)
  CASE 
    WHEN tenure < 12 AND MonthlyCharges < 50 THEN 'New Budget'
    WHEN tenure < 12 AND MonthlyCharges >= 50 THEN 'New Premium'
    WHEN tenure >= 36 AND MonthlyCharges < 50 THEN 'Loyal Budget'
    WHEN tenure >= 36 AND MonthlyCharges >= 50 THEN 'Loyal Premium'
    ELSE 'Regular'
  END AS customer_segment,
  
  -- 6. churn_label (binary for modeling)
  -- PERBAIKAN: Churn adalah BOOL (TRUE = 1, FALSE = 0)
  CASE WHEN Churn = TRUE THEN 1 ELSE 0 END AS churn_label

FROM base;
"""

print("🚀 Creating features table with Network KQI...")
print("📝 Using BOOL type for Churn (TRUE/FALSE)")
query_job = client.query(sql_create_features)
query_job.result()  # Wait for completion
print(f"✅ Table {TARGET_TABLE} created successfully!")

🚀 Creating features table with Network KQI...
📝 Using BOOL type for Churn (TRUE/FALSE)
✅ Table features_network_kqi created successfully!


In [6]:
# Verifikasi table berhasil dibuat
query_check = f"""
SELECT 
  COUNT(*) as total_rows,
  ROUND(AVG(avg_throughput_mbps), 1) as avg_throughput,
  ROUND(AVG(latency_ms), 1) as avg_latency,
  ROUND(AVG(network_quality_score), 1) as avg_quality,
  SUM(CASE WHEN churn_label = 1 THEN 1 ELSE 0 END) as churned_customers,
  ROUND(AVG(churn_label) * 100, 2) as churn_rate_pct
FROM `{PROJECT_ID}.{DATASET_ID}.{TARGET_TABLE}`
"""

result = client.query(query_check).to_dataframe()
print("✅ Table verification:")
result

/Users/macos/Study_burhanudin_2025/GCP/telco-ai-portfolio/venv_telco/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


✅ Table verification:


,total_rows,avg_throughput,avg_latency,avg_quality,churned_customers,churn_rate_pct
0,14086,41.2,54.7,42.2,3738,26.54


In [7]:
# Cell 3: Preview the data
query_preview = f"""
SELECT 
  customerID,
  tenure,
  InternetService,
  ROUND(avg_throughput_mbps, 1) as throughput,
  ROUND(latency_ms, 1) as latency,
  ROUND(drop_call_rate, 3) as drop_rate,
  ROUND(network_quality_score, 1) as quality_score,
  Churn
FROM `{PROJECT_ID}.{DATASET_ID}.{TARGET_TABLE}`
LIMIT 10
"""

df_preview = client.query(query_preview).to_dataframe()
df_preview

/Users/macos/Study_burhanudin_2025/GCP/telco-ai-portfolio/venv_telco/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,customerID,tenure,InternetService,throughput,latency,drop_rate,quality_score,Churn
0,8735-SDUFN,72,No,5.9,144.6,0.062,16.0,False
1,8735-SDUFN,72,No,6.3,147.1,0.071,16.0,False
2,5334-AFQJB,72,No,7.1,133.9,0.068,16.0,False
3,5334-AFQJB,72,No,7.0,104.7,0.098,16.0,False
4,5605-XNWEN,72,No,6.2,130.0,0.085,16.0,False
5,5605-XNWEN,72,No,7.8,142.2,0.103,16.0,False
6,0795-XCCTE,72,No,6.9,132.9,0.062,16.0,False
7,0795-XCCTE,72,No,8.9,108.8,0.073,16.0,False
8,2452-MRMZF,72,No,7.3,101.5,0.108,16.0,False
9,2452-MRMZF,72,No,7.5,140.2,0.098,16.0,False


In [8]:
# Cell 4: Check data distribution
query_stats = f"""
SELECT 
  COUNT(*) as total_customers,
  ROUND(AVG(avg_throughput_mbps), 2) as avg_throughput,
  ROUND(AVG(latency_ms), 2) as avg_latency,
  ROUND(AVG(drop_call_rate), 4) as avg_drop_rate,
  ROUND(AVG(network_quality_score), 2) as avg_quality,
  ROUND(AVG(churn_label) * 100, 2) as churn_rate_pct
FROM `{PROJECT_ID}.{DATASET_ID}.{TARGET_TABLE}`
"""

stats = client.query(query_stats).to_dataframe()
stats

/Users/macos/Study_burhanudin_2025/GCP/telco-ai-portfolio/venv_telco/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,total_customers,avg_throughput,avg_latency,avg_drop_rate,avg_quality,churn_rate_pct
0,14086,41.24,54.7,0.0984,42.21,26.54
